## Bronze to Silver: Data Cleansing and Transformation

In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

#### Create Widgets

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "stgcbpadlsdev101", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

### Stream Bronze Table in a Dataframe

In [0]:
df = spark.readStream \
.format("delta") \
.table(f"{catalog_name}.bronze.brz_order_items")

#display(df.limit(20), checkpointLocation=f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/display_preview/")

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,_rescued_data,ingest_timestamp,source_file
2025-08-30,2025-08-30T12:46:50.000Z,CUST000000114495,676395,1,2000000136295,1,GBP,13,16%,1,app,NEW10,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv
2025-08-30,2025-08-30T19:00:46.000Z,CUST000000167574,676396,1,2000000125442,1,AUD,23,7%,3,web,PRIME5,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv
2025-08-30,2025-08-30T19:00:46.000Z,CUST000000167574,676396,2,2000000319490,1,AUD,487,4%,24,web,PRIME5,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv
2025-08-30,2025-08-30T21:50:52.000Z,CUST000000091703,676397,1,2000000213705,1,INR,3199,2%,158,web,FEST20,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv
2025-08-30,2025-08-30T23:23:09.000Z,CUST000000027113,676398,1,2000000383569,1,INR,1696,16%,72,app,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv
2025-08-30,2025-08-30T04:52:11.000Z,CUST000000128388,676399,1,2000000333991,1,INR,31798,3%,5566,app,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv
2025-08-30,2025-08-30T16:11:21.000Z,CUST000000105911,676400,1,2000000017921,1,INR,12964,17%,1930,app,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv
2025-08-30,2025-08-30T17:17:40.000Z,CUST000000273320,676401,1,2000000275246,Two,GBP,294,0%,30,web,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv
2025-08-30,2025-08-30T10:26:35.000Z,CUST000000012580,676402,1,2000000013053,Two,INR,2171,13%,455,web,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv
2025-08-30,2025-08-30T10:26:35.000Z,CUST000000012580,676402,2,2000000005201,1,INR,1680,6%,286,web,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv


### Perform Transformations and Cleaning

In [0]:
df = df.dropDuplicates(["order_id", "item_seq"])

# Transformation: Convert 'Two' → 2 and cast to Integer
df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
)

# Transformation : Remove any '$' or other symbols from unit_price, keep only numeric
df = df.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

# Transformation : Remove '%' from discount_pct and cast to double
df = df.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

# Transformation : coupon code processing (convert to lower)
df = df.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

# Transformation : channel processing 
df = df.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

#Transformation : Add processed time 
df = df.withColumn(
    "processed_time", F.current_timestamp()
)


In [0]:
#display(df.limit(20), checkpointLocation=f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/display_preview1/")

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,_rescued_data,ingest_timestamp,source_file,processed_time
2025-08-30,2025-08-30T07:45:21.000Z,CUST000000038846,676515,1,2000000361970,1,CAD,80.0,8.0,4,Website,fest20,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z
2025-08-30,2025-08-30T19:25:38.000Z,CUST000000084592,677114,1,2000000203256,1,USD,84.0,10.0,14,Mobile,save50,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z
2025-08-30,2025-08-30T17:01:59.000Z,CUST000000293330,677240,2,2000000181189,1,SGD,33.0,0.0,7,Mobile,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z
2025-08-30,2025-08-30T18:01:23.000Z,CUST000000243652,677308,1,2000000388977,1,USD,14.0,11.0,1,Website,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z
2025-08-30,2025-08-30T22:19:03.000Z,CUST000000158878,677330,1,2000000070216,1,CAD,714.0,8.0,33,Mobile,prime5,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z
2025-08-30,2025-08-30T04:02:10.000Z,CUST000000275031,677473,1,2000000361543,2,INR,901.0,0.0,217,Website,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z
2025-08-30,2025-08-30T01:41:04.000Z,CUST000000232652,677551,1,2000000401744,1,AED,319.0,8.0,15,Website,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z
2025-08-30,2025-08-30T09:07:59.000Z,CUST000000045796,677858,2,2000000163628,1,USD,170.0,2.0,20,Website,save50,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z
2025-08-30,2025-08-30T00:24:09.000Z,CUST000000179323,677886,3,2000000185842,1,INR,71360.0,0.0,8564,Website,null,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z
2025-08-30,2025-08-30T07:56:26.000Z,CUST000000259824,677955,1,2000000059501,1,SGD,23.0,14.0,3,Website,prime5,null,2026-03-12T04:46:07.915Z,abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/order_items/landing/order_items_2025-08-30.csv,2026-03-12T04:56:42.462Z


### Save to Silver Table

In [0]:
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_items/"
print(silver_checkpoint_path)

abfss://ecomm-raw-data@stgcbpadlsdev101.dfs.core.windows.net/checkpoint/silver/fact_order_items/


In [0]:
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    

    



In [0]:
# This line is running a Structured Streaming job that:
# - Reads incremental data from Bronze (df).
# - For each batch → applies upsert_to_silver (update if exists, insert if not).
# - Writes into a Silver Delta table with schema evolution enabled.
# - Uses checkpointing for recovery.
# - Runs in batch-like mode (once or availableNow), not continuous streaming.

df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()